In [1]:
import torch
import pandas as pd
import numpy as np

# feature_selection
name_desc = pd.read_csv('https://han-attention.s3.amazonaws.com/new_task/data813/name_desc/name_desc_all_GPT4.csv')
emb_all = pd.read_csv('https://han-attention.s3.amazonaws.com/input/phenotype_RA/Final_naive_dropout.csv')

emb_all.shape

(24682, 769)

In [2]:
emb_MGB = emb_all.iloc[0:6917,1:]

In [7]:
emb_sap = pd.read_csv('https://han-attention.s3.amazonaws.com/input/emb/emb_Z0_825.csv')

emb_svd = emb_sap.iloc[0:6917,769:]
emb_sap = emb_sap.iloc[0:6917,1:769]

In [3]:
name_all = name_desc.iloc[0:6917,0].values

In [4]:
def feature_selection(emb_all, code, name_all, feat_max=100):
    # Find the row corresponding to the code
    emb_code_row = emb_all[name_all == code].values[0, 1:]
    # Convert the data to PyTorch tensors
    emb_all_tensor = torch.tensor(emb_all.values[:, 1:].astype(np.float32))
    emb_code_tensor = torch.tensor(emb_code_row.astype(np.float32))
    # Compute the cosine similarity
    cos_sim = torch.matmul(emb_all_tensor, emb_code_tensor) / (torch.norm(emb_all_tensor, dim=1) * torch.norm(emb_code_tensor))
    # Get the indices of the top feat_max features based on their cosine similarity
    top_indices = torch.topk(cos_sim, k=feat_max).indices.numpy()
    return top_indices



In [10]:
fea_sap = feature_selection(emb_sap, 'PheCode:714.1', name_all[0:6917])
name_desc.iloc[0:6917,:].iloc[fea_sap,0:2].to_csv('sap_temp_RA.csv')

fea_svd = feature_selection(emb_svd, 'PheCode:714.1', name_all[0:6917])
name_desc.iloc[0:6917,:].iloc[fea_svd,0:2].to_csv('svd_temp_RA.csv')

In [5]:
fea_all = feature_selection(emb_MGB, 'PheCode:714.1', name_all[0:6917])
name_desc.iloc[0:6917,:].iloc[fea_all,0:2].to_csv('gat_temp_RA.csv')